# ColumnTransformer

In [ ]:
import numpy as np
import pandas as pd

from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.preprocessing import OrdinalEncoder

In [ ]:
df = pd.read_csv('/content/covid_toy.csv')

In [ ]:
df.head()

,age,gender,fever,cough,city,has_covid
0,60,Male,103.0,Mild,Kolkata,No
1,27,Male,100.0,Mild,Delhi,Yes
2,42,Male,101.0,Mild,Delhi,No
3,31,Female,98.0,Mild,Kolkata,No
4,65,Female,101.0,Mild,Mumbai,No


In [ ]:
df.isnull().sum()

,0
age,0
gender,0
fever,10
cough,0
city,0
has_covid,0


In [ ]:
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(df.drop(columns=['has_covid']), df['has_covid'], test_size=0.2)

In [ ]:
X_train

,age,gender,fever,cough,city
97,20,Female,101.0,Mild,Bangalore
99,10,Female,98.0,Strong,Kolkata
11,65,Female,98.0,Mild,Mumbai
68,54,Female,104.0,Strong,Kolkata
93,27,Male,100.0,Mild,Kolkata
...,...,...,...,...,...
19,42,Female,NaN,Strong,Bangalore
20,12,Male,98.0,Strong,Bangalore
25,23,Male,NaN,Mild,Mumbai
98,5,Female,98.0,Strong,Mumbai


# 1. Without ColumnTransformer

In [ ]:
# adding simple impute to fever cost
si = SimpleImputer()

X_train_fever = si.fit_transform(X_train[['fever']])

X_test_fever = si.transform(X_test[['fever']])

print(X_train_fever.shape)
print(X_test_fever.shape)

(80, 1)
(20, 1)


In [ ]:
# ordinal encoding - cough
oe = OrdinalEncoder(categories=[['Mild', 'Strong']])

X_train_cough = oe.fit_transform(X_train[['cough']])

X_test_cough = oe.transform(X_test[['cough']])

print(X_train_cough.shape)
print(X_test_cough.shape)

(80, 1)
(20, 1)


In [ ]:
# OneHotEncoding - gender, city
ohe = OneHotEncoder(drop='first', sparse_output=False)

X_train_gender_city = ohe.fit_transform(X_train[['gender', 'city']])

X_test_gender_city = ohe.transform(X_test[['gender', 'city']])

print(X_train_gender_city.shape)
print(X_test_gender_city.shape)

(80, 4)
(20, 4)


In [ ]:
# Extracting age
X_train_age = X_train.drop(columns=['gender', 'fever', 'cough', 'city']).values

X_test_age = X_test.drop(columns = ['gender', 'fever', 'cough', 'city']).values

print(X_train_age.shape)
print(X_test_age.shape)

(80, 1)
(20, 1)


In [ ]:
X_train_transformed = np.concatenate((X_train_age, X_train_fever, X_train_gender_city, X_train_cough), axis = 1)

X_test_transformed = np.concatenate((X_test_age, X_test_fever, X_test_gender_city, X_test_cough), axis = 1)

print(X_train_transformed.shape)
print(X_test_transformed.shape)

(80, 7)
(20, 7)


# 2. With ColumnTransformer


In [ ]:
from sklearn.compose import ColumnTransformer

In [ ]:
transformer = ColumnTransformer(transformers = [
    ('tnf1', SimpleImputer(), ['fever']),
    ('tnf2', OrdinalEncoder(categories=[['Mild', 'Strong']]), ['cough']),
    ('tnf3', OneHotEncoder(sparse_output=False, drop='first'), ['gender', 'city'])
], remainder='passthrough')

In [ ]:
transformer.fit_transform(X_train).shape

(80, 7)

In [ ]:
transformer.fit_transform(X_test).shape

(20, 7)